# Pack local safetensors → SRD-4 .axm + GGUF

Packs any number of locally-downloaded models (Qwen, Gemma, or any causal LM)
into signed `.axm` governance containers with SRD-4 quantization, then extracts
GGUF files for llama.cpp inference.

**What you need:**
- Each model as a local directory containing `config.json` + `model.safetensors`
- GPU with ≥ 6 GB VRAM (or CPU — slower)
- No HuggingFace account or internet for the models (already on disk)

**Steps:**
1. `Cell 1` — GPU check + pip install
2. `Cell 2` — Clone axiom repo, set up `AXIOM_MASTER_KEY`
3. `Cell 3` — ⚙️ **Fill in your model paths here**
4. `Cell 4` — SRD pack all models → `.axm`
5. `Cell 5` — Verify HMAC proof chains
6. `Cell 6` — Download pre-built llama.cpp + extract GGUFs
7. `Cell 7` — Summary table

In [ ]:
# Cell 1 — GPU check + dependencies
import subprocess, sys, os

try:
    import torch
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f"GPU : {p.name}")
        print(f"VRAM: {p.total_memory/1024**3:.1f} GB")
    else:
        print("No GPU detected — will run on CPU (slower)")
except ImportError:
    print("PyTorch not installed yet — will install below")

print("\nInstalling dependencies...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.45",
    "accelerate",
    "bitsandbytes",
    "safetensors",
    "sentencepiece",
    "protobuf",
    "psutil",
    "tqdm",
], check=True)
print("✓ dependencies ready")

In [ ]:
# Cell 2 — Clone axiom repo + persist AXIOM_MASTER_KEY
import os, secrets, subprocess, sys
from pathlib import Path

# ── Paths (change if not on Colab) ────────────────────────────────────────────
AXIOM_DIR   = Path("/content/axiom")         # RunPod: /workspace/axiom
OUTPUT_DIR  = Path("/content/axm_output")    # RunPod: /workspace/axm_output
LLAMACPP    = Path("/content/llama.cpp")     # RunPod: /workspace/llama.cpp
BRANCH      = "claude/srd-prototype-benchmark-JRtv1"
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Clone axiom if not present
if not AXIOM_DIR.is_dir():
    print(f"Cloning axiom ({BRANCH})...")
    subprocess.run([
        "git", "clone", "--branch", BRANCH, "--depth", "1",
        "https://github.com/orivael-dev/axiom.git", str(AXIOM_DIR),
    ], check=True)
    print("✓ axiom cloned")
else:
    print(f"✓ axiom already at {AXIOM_DIR}")

sys.path.insert(0, str(AXIOM_DIR))

# Persist AXIOM_MASTER_KEY — NEVER regenerate mid-session or AXM signatures break
KEY_FILE = OUTPUT_DIR / "axiom_master.key"
if os.environ.get("AXIOM_MASTER_KEY"):
    print("AXIOM_MASTER_KEY: from environment")
elif KEY_FILE.is_file():
    os.environ["AXIOM_MASTER_KEY"] = KEY_FILE.read_text().strip()
    print(f"AXIOM_MASTER_KEY: restored from {KEY_FILE}")
else:
    key = secrets.token_hex(32)
    os.environ["AXIOM_MASTER_KEY"] = key
    KEY_FILE.write_text(key)
    print(f"AXIOM_MASTER_KEY: generated and saved to {KEY_FILE}")
    print("  ⚠ Back this file up — you need it to verify these AXMs later")

print(f"\nOutput directory: {OUTPUT_DIR}")

In [ ]:
# Cell 3 — ⚙️  CONFIGURE YOUR MODEL PATHS HERE
#
# Each entry: "short_name": {"path": "/absolute/path/to/model/dir", "quant": "Q4_K_M"}
#   path  — directory containing config.json + model.safetensors (or shards)
#   quant — GGUF quantization for llama.cpp: Q4_K_M | Q5_K_M | Q6_K | F16
#
MODELS = {
    "qwen_1": {
        "path":  "/path/to/your/qwen_model_1",  # ← CHANGE THIS
        "quant": "Q4_K_M",
    },
    "qwen_2": {
        "path":  "/path/to/your/qwen_model_2",  # ← CHANGE THIS
        "quant": "Q4_K_M",
    },
    "gemma": {
        "path":  "/path/to/your/gemma_model",   # ← CHANGE THIS
        "quant": "Q4_K_M",
    },
}

# Validate paths exist
ok = True
for name, cfg in MODELS.items():
    p = Path(cfg["path"])
    has_cfg    = (p / "config.json").exists()
    has_st     = any(p.glob("*.safetensors"))
    has_bin    = any(p.glob("*.bin"))
    status = "✓" if (has_cfg and (has_st or has_bin)) else "✗"
    fmt    = "safetensors" if has_st else ("pytorch_bin" if has_bin else "MISSING WEIGHTS")
    print(f"{status} [{name:10s}]  {p}  ({fmt})")
    if status == "✗":
        ok = False

if not ok:
    raise SystemExit("Fix the paths above before continuing")
print("\nAll model paths validated ✓")

In [ ]:
# Cell 4 — SRD-4 pack all models → signed .axm
#
# Each model is quantized with SRD (Stochastic Residual Dithering, ~4.5 bpw)
# and packed into a signed .axm container.
# safe_serialization=True is enforced in pack_to_axm.py — weights stored as
# safetensors inside the archive for zero-copy verified loading.

import json, subprocess, sys, time
from pathlib import Path

PACK_SCRIPT = AXIOM_DIR / "research" / "quant" / "pack_to_axm.py"
RESULTS     = {}   # name → {axm_path, fingerprint, bpw, pack_min, ...}

for name, cfg in MODELS.items():
    print(f"\n{'='*60}")
    print(f"Packing: {name}  ({cfg['path']})")
    print(f"{'='*60}")

    model_out = OUTPUT_DIR / name
    model_out.mkdir(exist_ok=True)
    axm_out   = model_out / f"{name}_srd4.axm"
    stats_f   = model_out / "pack_stats.json"

    t0 = time.time()
    result = subprocess.run(
        [
            sys.executable, str(PACK_SCRIPT),
            "--model",      cfg["path"],
            "--srd-top-k-pct", "0.25",   # ~4.5 bpw
            "--output",     str(axm_out),
        ],
        cwd=str(AXIOM_DIR),
    )
    elapsed = time.time() - t0

    if result.returncode != 0:
        print(f"  ✗ FAILED (return code {result.returncode})")
        RESULTS[name] = {"status": "FAILED", "axm_path": None}
        continue

    size_gb = axm_out.stat().st_size / 1024**3 if axm_out.exists() else 0
    print(f"  ✓ {axm_out.name}  ({size_gb:.3f} GB)  {elapsed/60:.1f} min")

    RESULTS[name] = {
        "status":    "packed",
        "axm_path":  str(axm_out),
        "axm_gb":    round(size_gb, 3),
        "pack_min":  round(elapsed / 60, 1),
        "quant":     cfg["quant"],
    }

print("\nPacking complete.")
for name, r in RESULTS.items():
    print(f"  {r['status']:8s}  {name:10s}  {r.get('axm_gb',''):>6} GB  {r.get('pack_min',''):>4} min")

In [ ]:
# Cell 5 — Verify HMAC proof chains
#
# Confirms every tensor block was signed correctly at pack time.
# A broken signature means the archive was tampered with.

import json, subprocess, sys
from pathlib import Path

AXM_CLI = AXIOM_DIR / "axm_cli.py"

for name, r in RESULTS.items():
    if not r.get("axm_path"):
        print(f"  SKIP  {name}  (not packed)")
        continue

    out = subprocess.run(
        [sys.executable, str(AXM_CLI), "verify", r["axm_path"]],
        cwd=str(AXIOM_DIR), capture_output=True, text=True,
    )
    try:
        data = json.loads(out.stdout)
    except Exception:
        data = {"verified": False, "error": out.stdout + out.stderr}

    ok = data.get("verified", False)
    fp = data.get("fingerprint", "?")
    proofs = data.get("proofs_checked", "?")

    mark = "✓" if ok else "✗"
    print(f"  {mark} [{name:10s}]  fingerprint={fp}  proofs={proofs}")

    RESULTS[name]["verified"]    = ok
    RESULTS[name]["fingerprint"] = fp
    RESULTS[name]["proofs"]      = proofs

    if not ok:
        print(f"     ERROR: {data.get('error', 'unknown')}")

In [ ]:
# Cell 6 — Download pre-built llama.cpp binary + extract GGUFs
#
# Downloads the latest pre-built ubuntu-x64 CUDA binary from the llama.cpp
# GitHub releases — no compile, ~2 minutes download vs 15 minutes build.
# Falls back to source build if the pre-built binary is unavailable.

import io, json, os, subprocess, sys, time, urllib.request, zipfile
from pathlib import Path

def _get_llamacpp(llamacpp_dir: Path) -> Path:
    """Return path to llama-quantize binary, downloading pre-built if needed."""
    quantize = llamacpp_dir / "build" / "bin" / "llama-quantize"
    if quantize.is_file():
        print(f"  llama-quantize: already at {quantize}")
        return quantize

    # Clone repo (needed for convert scripts even with pre-built binary)
    if not llamacpp_dir.is_dir():
        print("  Cloning llama.cpp...")
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/ggerganov/llama.cpp.git",
                        str(llamacpp_dir)], check=True)

    # Try pre-built CUDA binary from GitHub releases
    try:
        import torch
        cuda_ver = torch.version.cuda or ""
        major_minor = ".".join(cuda_ver.split(".")[:2])
    except Exception:
        major_minor = ""

    print(f"  Fetching pre-built llama.cpp release (CUDA {major_minor or 'unknown'})...")
    try:
        req = urllib.request.Request(
            "https://api.github.com/repos/ggerganov/llama.cpp/releases/latest",
            headers={"User-Agent": "axiom-pack/1.0"},
        )
        with urllib.request.urlopen(req, timeout=20) as r:
            release = json.loads(r.read())
        assets = release.get("assets", [])

        def _score(a):
            n = a["name"]
            if "ubuntu-x64" not in n or "cuda" not in n: return 0
            return 2 if (major_minor and f"cu{major_minor}" in n) else 1

        best = max(assets, key=_score, default=None)
        if best and _score(best) > 0:
            mb = best["size"] / 1024**2
            print(f"  Downloading {best['name']}  ({mb:.0f} MB)...")
            with urllib.request.urlopen(best["browser_download_url"], timeout=300) as r:
                data = r.read()
            build_bin = llamacpp_dir / "build" / "bin"
            build_bin.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(io.BytesIO(data)) as z:
                for member in z.namelist():
                    stem = Path(member).name
                    if stem in ("llama-quantize", "llama-cli", "llama-convert-hf-to-gguf"):
                        dest = build_bin / stem
                        dest.write_bytes(z.read(member))
                        dest.chmod(0o755)
                        print(f"    ✓ {stem}")
            if quantize.is_file():
                print("  llama-quantize: pre-built binary ready")
                return quantize
    except Exception as e:
        print(f"  Pre-built download failed: {e}")

    # Fallback: cmake build
    print("  Building from source (cmake -DGGML_CUDA=ON)...")
    try:
        import torch
        p = torch.cuda.get_device_properties(0)
        arch = f"{p.major}{p.minor}"
    except Exception:
        arch = "native"
    subprocess.run([
        "cmake", "-B", str(llamacpp_dir / "build"), "-S", str(llamacpp_dir),
        "-DGGML_CUDA=ON", f"-DCMAKE_CUDA_ARCHITECTURES={arch}",
        "-DCMAKE_BUILD_TYPE=Release",
    ], check=True)
    subprocess.run([
        "cmake", "--build", str(llamacpp_dir / "build"),
        "-j", "4", "--target", "llama-quantize", "llama-cli",
    ], check=True, timeout=900)
    print("  llama-quantize: built from source")
    return quantize


# Get llama.cpp
QUANTIZE_BIN = _get_llamacpp(LLAMACPP)

# Extract GGUFs for each packed model
AXM_CLI = AXIOM_DIR / "axm_cli.py"

for name, r in RESULTS.items():
    if not r.get("axm_path") or not r.get("verified"):
        print(f"  SKIP  {name}  ({'not packed' if not r.get('axm_path') else 'verify failed'})")
        continue

    axm  = Path(r["axm_path"])
    gguf = axm.with_name(f"{name}_srd4_{r['quant'].lower()}.gguf")

    print(f"\n{'─'*50}")
    print(f"Extracting GGUF: {name}  ({r['quant']})")

    t0 = time.time()
    result = subprocess.run(
        [
            sys.executable, str(AXM_CLI), "extract", str(axm),
            "--gguf-out",  str(gguf),
            "--llamacpp",  str(LLAMACPP),
            "--quant",     r["quant"],
            "--device",    "cpu",
        ],
        cwd=str(AXIOM_DIR),
    )
    elapsed = time.time() - t0

    if result.returncode != 0 or not gguf.exists():
        print(f"  ✗ extraction failed")
        RESULTS[name]["gguf_path"] = None
        continue

    gguf_gb = gguf.stat().st_size / 1024**3
    print(f"  ✓ {gguf.name}  ({gguf_gb:.3f} GB)  {elapsed/60:.1f} min")
    RESULTS[name]["gguf_path"] = str(gguf)
    RESULTS[name]["gguf_gb"]   = round(gguf_gb, 3)
    RESULTS[name]["gguf_min"]  = round(elapsed / 60, 1)

In [ ]:
# Cell 7 — Summary table

import json
from pathlib import Path

print("\n" + "="*80)
print("PACK RESULTS")
print("="*80)
print(f"{'Model':<12} {'Status':<8} {'AXM':>8} {'GGUF':>8} {'Fingerprint':<12} {'Proofs':>6} {'Pack min':>8} {'GGUF min':>8}")
print("-"*80)

total_axm_gb = 0.0
for name, r in RESULTS.items():
    axm_gb   = r.get("axm_gb", 0) or 0
    gguf_gb  = r.get("gguf_gb", 0) or 0
    total_axm_gb += axm_gb
    print(
        f"{name:<12} "
        f"{'✓' if r.get('verified') else ('packed' if r.get('axm_path') else '✗'):<8} "
        f"{axm_gb:>7.3f}G "
        f"{gguf_gb:>7.3f}G "
        f"{r.get('fingerprint','?'):<12} "
        f"{str(r.get('proofs','?')):>6} "
        f"{r.get('pack_min',0):>8.1f} "
        f"{r.get('gguf_min',0):>8.1f}"
    )

print("-"*80)
print(f"Total .axm disk: {total_axm_gb:.3f} GB")

# Save consolidated results JSON
results_path = OUTPUT_DIR / "pack_results.json"
results_path.write_text(json.dumps(RESULTS, indent=2))
print(f"\nResults JSON: {results_path}")

print("\n--- Artifacts ---")
for name, r in RESULTS.items():
    if r.get("axm_path"):
        print(f"  .axm  {r['axm_path']}")
    if r.get("gguf_path"):
        print(f"  .gguf {r['gguf_path']}")

print("\n--- To run inference with llama.cpp ---")
for name, r in RESULTS.items():
    if r.get("gguf_path"):
        print(f"  # {name}")
        print(f"  {LLAMACPP}/build/bin/llama-cli -m {r['gguf_path']} --ngl 99 -p 'Hello' -n 64")